<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/ex_frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install keras keras-hub --upgrade -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

Keras 3 peut s’exécuter sur plusieurs backends (TF / Torch / JAX) ; il faut définir le backend avant les imports.

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

### TensorFlow — Tenseurs constants

Avec TensorFlow :
- créez un tenseur rempli de 1 de forme `(2, 1)`,
- créez un tenseur rempli de 0 de forme `(2, 1)`,
- créez un tenseur constant contenant `[1, 2, 3]` en `float32`.

### TensorFlow — Tenseurs aléatoires

Générez et affichez :
- un tenseur aléatoire suivant une loi normale de forme `(3, 1)` (moyenne 0, écart-type 1),
- un tenseur aléatoire uniforme de forme `(3, 1)` (valeurs entre 0 et 1).

In [ ]:
x = tf.random.normal(shape=(3, 1), mean=0., stddev=1.)
print(x)

In [ ]:
x = tf.random.uniform(shape=(3, 1), minval=0., maxval=1.)
print(x)

### TensorFlow — Variables et affectation

1) Montrez qu’avec NumPy on peut modifier un tableau “en place” (ex : mettre à 0 l’élément `[0,0]` d’un tableau de 1).

2) Créez une `tf.Variable` initialisée avec un tenseur aléatoire de forme `(3,1)` puis :
- remplacez sa valeur par un tenseur de 1 (`assign`),
- modifiez un seul élément (ex : `[0,0]`) avec `assign`,
- ajoutez 1 partout (`assign_add`).

### TensorFlow — Opérations de base et couche dense

1) À partir d’un tenseur `a = ones((2,2))`, calculez :
- `b = square(a)`, `c = sqrt(a)`,
- `d = b + c`,
- `e = matmul(a, b)`,
- `f = concat((a, b), axis=0)`.

2) Écrivez une fonction `dense(inputs, W, b)` qui renvoie `relu(matmul(inputs, W) + b)`.

### TensorFlow — Calcul automatique de gradients (GradientTape)

1) Créez une variable scalaire `input_var = 3.0` et calculez le gradient de `input_var^2` par rapport à `input_var`.

2) Recommencez avec une constante `input_const = 3.0` (pensez à `tape.watch`).

3) Utilisez deux `GradientTape` imbriquées pour :
- définir `position = 4.9 * time^2`,
- obtenir la vitesse (dérivée de la position),
- puis l’accélération (dérivée de la vitesse).

In [ ]:
input_var = tf.Variable(initial_value=3.0)
with tf.GradientTape() as tape:
    result = tf.square(input_var)
gradient = tape.gradient(result, input_var)

In [ ]:
input_const = tf.constant(3.0)
with tf.GradientTape() as tape:
    tape.watch(input_const)
    result = tf.square(input_const)
gradient = tape.gradient(result, input_const)

In [ ]:
time = tf.Variable(0.0)
with tf.GradientTape() as outer_tape:
    with tf.GradientTape() as inner_tape:
        position = 4.9 * time**2
    speed = inner_tape.gradient(position, time)
acceleration = outer_tape.gradient(speed, time)

### TensorFlow — Accélérer une fonction avec compilation

Redéfinissez la fonction `dense(inputs, W, b)` en l’annotant avec :
- `@tf.function`,
puis avec :
- `@tf.function(jit_compile=True)`.

In [ ]:
@tf.function
def dense(inputs, W, b):
    return tf.nn.relu(tf.matmul(inputs, W) + b)

In [ ]:
@tf.function(jit_compile=True)
def dense(inputs, W, b):
    return tf.nn.relu(tf.matmul(inputs, W) + b)

### À quoi servent `@tf.function` et `jit_compile=True` ?

#### 🔹 `@tf.function`

- Transforme une fonction TensorFlow écrite en Python en **graphe de calcul optimisé**.
- Passe du mode *eager* (exécution ligne par ligne) au **mode graph**.
- Permet :
  - une exécution plus rapide,
  - des optimisations automatiques,
  - de meilleures performances sur GPU/TPU.
- Particulièrement utile dans les **boucles d’entraînement**.

---

#### 🔹 `jit_compile=True`

- Active la compilation **XLA (Accelerated Linear Algebra)**.
- Ajoute une couche d’optimisation supplémentaire au graphe TensorFlow.
- Peut :
  - fusionner des opérations,
  - réduire les accès mémoire,
  - accélérer fortement certains calculs.

---

| Décorateur | Rôle |
|------------|------|
| `@tf.function` | Compile en graphe TensorFlow optimisé |
| `jit_compile=True` | Ajoute une optimisation XLA pour plus de performance |

---

⚠️ À éviter principalement pendant le débogage (messages d’erreur moins lisibles).

### TensorFlow — Exemple complet : génération d’un dataset 2D

Générez un jeu de données 2D synthétique pour une classification binaire :
- 1000 points pour une classe “négative” tirés d’une gaussienne multivariée,
- 1000 points pour une classe “positive” tirés d’une autre gaussienne multivariée,
avec des moyennes et covariances différentes.

In [ ]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)

In [ ]:
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

### TensorFlow — Préparer les tenseurs d’entrée et les labels

1) Concaténez les deux nuages de points en une matrice `inputs` de type `float32`.

2) Construisez `targets` sous forme d’un vecteur colonne :
- des 0 pour la première classe,
- des 1 pour la deuxième.

In [ ]:
targets = np.vstack(
    (
        np.zeros((num_samples_per_class, 1), dtype="float32"),
        np.ones((num_samples_per_class, 1), dtype="float32"),
    )
)

### TensorFlow — Visualisation du dataset

Tracez le nuage de points en 2D avec `matplotlib`, en colorant les points selon leur classe (`targets`).

### TensorFlow — Initialisation du modèle linéaire

Définissez :
- `input_dim = 2`, `output_dim = 1`,
- une matrice de poids `W` (variable TF) initialisée aléatoirement,
- un biais `b` (variable TF) initialisé à 0.

In [ ]:
input_dim = 2
output_dim = 1
W = tf.Variable(initial_value=tf.random.uniform(shape=(input_dim, output_dim)))
b = tf.Variable(initial_value=tf.zeros(shape=(output_dim,)))

### TensorFlow — Modèle et fonction de coût

1) Écrivez `model(inputs, W, b)` qui calcule `matmul(inputs, W) + b`.

2) Écrivez `mean_squared_error(targets, predictions)` qui renvoie la moyenne de `(targets - predictions)^2`.

In [ ]:
def mean_squared_error(targets, predictions):
    per_sample_losses = tf.square(targets - predictions)
    return tf.reduce_mean(per_sample_losses)

### TensorFlow — Boucle d’entraînement (training_step)

Écrivez une fonction `training_step(inputs, targets, W, b)` (compilée avec `@tf.function(jit_compile=True)`) qui :
- calcule les prédictions,
- calcule la perte MSE,
- calcule les gradients de la perte par rapport à `W` et `b`,
- met à jour `W` et `b` par descente de gradient avec un `learning_rate = 0.1`,
- renvoie la perte.

In [ ]:
learning_rate = 0.1

@tf.function(jit_compile=True)
def training_step(inputs, targets, W, b):
    with tf.GradientTape() as tape:
        predictions = model(inputs, W, b)
        loss = mean_squared_error(predictions, targets)
    grad_loss_wrt_W, grad_loss_wrt_b = tape.gradient(loss, [W, b])
    W.assign_sub(grad_loss_wrt_W * learning_rate)
    b.assign_sub(grad_loss_wrt_b * learning_rate)
    return loss

### TensorFlow — Entraîner le modèle

Exécutez `training_step` pendant 40 itérations et affichez la perte à chaque étape (format lisible).

### TensorFlow — Visualiser la frontière de décision (via les prédictions)

1) Calculez `predictions = model(inputs, W, b)`.

2) Tracez à nouveau le nuage de points en colorant selon la classe prédite (`pred > 0.5`).

### TensorFlow — Tracer la droite de décision du classifieur linéaire

À partir de `W` et `b`, calculez l’équation de la droite où la sortie vaut 0.5 et tracez-la sur le nuage de points.

In [ ]:
x = np.linspace(-1, 4, 100)
y = -W[0] / W[1] * x + (0.5 - b) / W[1]
plt.plot(x, y, "-r")
plt.scatter(inputs[:, 0], inputs[:, 1], c=predictions[:, 0] > 0.5)

### PyTorch — Tenseurs constants

Avec PyTorch :
- créez un tenseur de 1 de taille `(2,1)`,
- un tenseur de 0 de taille `(2,1)`,
- un tenseur `[1,2,3]` en `float32`.

### PyTorch — Tenseurs aléatoires

Générez :
- un tenseur normal (moyenne 0, écart-type 1) de forme `(3,1)` en construisant explicitement `mean` et `std`,
- un tenseur uniforme `rand(3,1)`.

### PyTorch — Affectation et paramètres

1) Montrez une affectation “en place” sur un tenseur (`x[0,0] = 1`).

2) Créez un objet `torch.nn.parameter.Parameter` à partir d’un tenseur `x` (ex : un tenseur de zéros).

### PyTorch — Opérations de base et couche dense

1) Reproduisez les opérations : square, sqrt, addition, matmul, concat (cat dim=0) sur des tenseurs 2x2.

2) Écrivez une fonction `dense(inputs, W, b)` qui calcule `relu(matmul(inputs, W) + b)`.

### PyTorch — Calcul de gradients

1) Créez un scalaire `input_var=3.0` avec `requires_grad=True`.
2) Calculez `result = input_var^2`, appelez `backward()` et récupérez `input_var.grad`.
3) Montrez que refaire `backward()` sans remise à zéro accumule le gradient.
4) Réinitialisez le gradient (mettre `grad` à `None`).

### PyTorch — Exemple complet : classifieur linéaire

1) Initialisez `W` et `b` avec `requires_grad=True`.
2) Écrivez `model(inputs, W, b)` et `mean_squared_error(targets, predictions)`.
3) Écrivez `training_step(inputs, targets, W, b)` qui :
- calcule la loss,
- fait `backward()`,
- met à jour `W` et `b` avec un pas de gradient (sans tracking de gradient via `torch.no_grad()`),
- remet `W.grad` et `b.grad` à `None`,
- renvoie la loss.

### PyTorch — Créer un modèle en héritant de `torch.nn.Module`

Implémentez une classe `LinearModel` qui :
- hérite de `torch.nn.Module`,
- déclare `W` et `b` comme `torch.nn.Parameter`,
- définit `forward(self, inputs)` pour calculer `matmul(inputs, W) + b`.

In [ ]:
class LinearModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.W = torch.nn.Parameter(torch.rand(input_dim, output_dim))
        self.b = torch.nn.Parameter(torch.zeros(output_dim))

    def forward(self, inputs):
        return torch.matmul(inputs, self.W) + self.b

### PyTorch — Entraînement avec un optimizer

1) Instanciez le modèle et calculez une sortie sur des entrées converties en tenseurs torch.

2) Créez un optimizer `torch.optim.SGD` sur `model.parameters()`.

3) Écrivez un `training_step(inputs, targets)` qui :
- calcule la loss,
- fait `backward()`,
- appelle `optimizer.step()`,
- remet les gradients à zéro (`model.zero_grad()`),
- renvoie la loss.

### PyTorch — Accélérer via compilation

1) Créez une version compilée du modèle avec `torch.compile(model)`.

2) Décorez une fonction `dense` avec `@torch.compile` pour compiler son exécution.

`@torch.compile` permet de **compiler et optimiser automatiquement** un modèle ou une fonction PyTorch afin d’en **accélérer l’exécution**.

---

#### 🔹 Pourquoi ?

Par défaut, PyTorch fonctionne en mode **dynamique (eager execution)** :
- chaque opération est exécutée immédiatement,
- c’est très flexible et facile à déboguer,
- mais pas toujours optimal en performance.

`torch.compile` analyse le code, construit une représentation optimisée, puis génère une version plus rapide.

---

#### 🔹 Ce que ça fait concrètement

- Capture le graphe d’exécution dynamique
- Applique des optimisations (fusion d’opérations, réduction des accès mémoire…)
- Génère un code optimisé (souvent via TorchDynamo + AOTAutograd + backends comme Inductor)

---

#### 🔹 Comment l’utiliser ?

**Compiler un modèle :**

```python
model = torch.compile(model)

L’approche PyTorch se distingue principalement par :

- **Un modèle de calcul dynamique (define-by-run)**  
  Le graphe de calcul est construit dynamiquement à chaque exécution.  
  Cela rend le code plus intuitif, plus flexible et plus facile à déboguer.

- **Une intégration naturelle avec Python**  
  PyTorch se comporte comme une extension de NumPy avec support GPU et autograd.  
  Le code ressemble à du Python standard, sans couche d’abstraction complexe.

- **Un système de différentiation automatique simple et explicite**  
  Les gradients sont calculés via `backward()` et stockés dans les attributs `.grad`.  
  Le contrôle du cycle :
  - calcul de la loss  
  - `backward()`  
  - mise à jour des paramètres  
  - remise à zéro des gradients  
  est très transparent.

- **Une architecture modulaire avec `nn.Module`**  
  Les modèles combinent :
  - les paramètres (state),
  - le calcul (méthode `forward`),
  ce qui rend la structuration des réseaux claire et extensible.

- **Une philosophie orientée recherche et expérimentation**  
  Grâce à sa flexibilité et à son caractère impératif, PyTorch est particulièrement adapté au prototypage rapide et aux architectures complexes.